# 05. Visual Conditioning: reference tails, masks, роли и RoPE

<!-- study-navigation:start -->
**Основа:** [MIT 6.S184 / 2026](https://diffusion.csail.mit.edu/2026/index.html) · [Основной конспект](https://diffusion.csail.mit.edu/2026/docs/lecture_notes.pdf) · [Учебная карта](../../curriculum/README.md) · [Полное содержание](../../curriculum/CONTENTS.md).

**Связанные лекции:** [Lecture 4](https://diffusion.csail.mit.edu/2026/docs/20260128_Lecture_04_edited.pdf).
<!-- study-navigation:end -->

Исполним каждый метод visual_conditioning.py на batched/packed tensor layouts и отделим storage positions от semantic temporal positions.

[Маршрут](../README.md) · [Точные задания MIT](../MIT_BRIDGE.md) · [Методы и source map](../SOURCE_MAP.md) · [Измерения](../reports/VALIDATION.md).

Каждый опыт: **теория → формула → формы/единицы → прогноз → код → измерение → ограниченный вывод**. CPU по умолчанию; без pretrained weights, сети и ComfyUI jobs.


**Связь с MIT:** MIT Lab 3 Part 4–5 объясняет latent space и frozen codec, но reference-tail layouts/role IDs/semantic RoPE не задаются исходным MNIST lab. Сначала [общий VAE маршрут](../../vae/README.md).

[MIT course](https://diffusion.csail.mit.edu/2026/index.html) · [Основной конспект](https://diffusion.csail.mit.edu/2026/docs/lecture_notes.pdf).


In [ ]:
from pathlib import Path
import sys,math,json,copy
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
COURSE=next((p/'dit' for p in [Path.cwd(),*Path.cwd().parents] if (p/'dit/labkit/core.py').is_file()),None)
assert COURSE is not None,'Clone the study repository first; see dit/README.md (Colab setup).'
sys.path.insert(0,str(COURSE))
from labkit.core import (native,tiny_model,tiny_inputs,activate_gates,fresh_forward,
  load_k5_block,manual_single,manual_fused,CountingLinearField,tiny_bundle,
  clone_bundle,scalar_bundle,vector_from_bundle,dummy_text,show_source,
  check_sources,save_metrics,figure_path)
torch.set_num_threads(2);torch.manual_seed(42)
LAB='05_visual_conditioning_references_and_rope';metrics={}
print('Pinned source files checked:',check_sources())


## 1. Три схемы visual condition

Для physical C_v channels модель с visual_cond=True принимает 2C_v+1:

$$
X_{in}=\mathrm{concat}(X,C_{aux},M),\qquad
X_{in}\in\mathbb{R}^{B\times T\times H\times W\times(2C_v+1)}.
$$

`pretrain`: clean first frame в auxiliary channels на старте; noisy X не перезаписывается. `i2v`: X first frame перезаписывается clean latent, auxiliary channels нулевые, mask помечает known frame. `tail_cond_first_frame`: clean reference append в конец storage sequence; source восстанавливает его и маркирует mask.

Mask channel — условие для сети; generated_mask/role IDs — отдельные tensors с другой задачей. `visual_token_type_embeddings` добавляет learned role embedding после patch embedding. Канальный mask не заменяет attention padding mask и не описывает time σ.


In [ ]:
vc=native('core.algo.visual_conditioning');loop=native('core.algo.denoise_loop')
B,T,H,W,C=2,3,2,2,4
video=torch.randn(B,T,H,W,C);first=torch.randn(B,H,W,C);cu=torch.arange(B+1,dtype=torch.int32)*T
pre=loop._build_video_input(video.clone(),True,first,cu,'pretrain')
i2v=loop._build_video_input(video.clone(),True,first,cu,'i2v')
assert pre.shape==i2v.shape==(B,T,H,W,2*C+1)
assert torch.equal(pre[...,:C],video) and torch.equal(pre[:,0,...,C:2*C],first)
assert torch.equal(i2v[:,0,...,:C],first) and i2v[...,C:2*C].eq(0).all()
assert pre[:,0,-1,-1,-1].eq(1).all() and pre[:,1:,...,-1].eq(0).all()
metrics['visual_input_channels']={'physical':C,'model_input':2*C+1,'shape':list(pre.shape)}


## 2. Append / restore / generated mask

При R=1 или 2 refs длина становится T+R, generated frames по-прежнему T. Role=0 для generated, role=1 для references. Packed layout объединяет B и T; cu_seqlens[b+1]−R даёт начало references каждого элемента. Нельзя взять последние R frames всего packed tensor: так будет испорчен только последний batch item.

`restore_tail_references` меняет переданный tensor in-place. `build_tail_condition` сначала восстанавливает refs, затем concat [video,zeros,mask]. Передайте clone, если нужен baseline до модификации. Ниже сравним batched и packed вычисления для B=2,R=2, где ошибка индексации была бы заметна.


In [ ]:
refs=torch.randn(B,2,H,W,C)
batched,roles,generated=vc.append_visual_references(video.clone(),refs,B,T)
packed,roles_p,generated_p=vc.append_visual_references(video.flatten(0,1).clone(),refs,B,T)
assert torch.equal(batched.flatten(0,1),packed)
assert roles.reshape(B,T+2)[:,-2:].eq(1).all() and generated.sum()==B*T
assert torch.equal(roles.flatten(),roles_p.flatten())
cu_ext=torch.arange(B+1,dtype=torch.int32)*(T+2)
corrupted=packed+10;vc.restore_tail_references(corrupted,refs,cu_ext)
assert torch.equal(corrupted.reshape(B,T+2,H,W,C)[:,-2:],refs)
inp=vc.build_tail_condition(packed.clone(),refs,cu_ext)
assert inp.shape[-1]==2*C+1
assert inp.reshape(B,T+2,H,W,-1)[:,-2:,...,-1].eq(1).all()
metrics['two_reference_contract']={'batched':list(batched.shape),'packed':list(packed.shape),'generated_frames':int(generated.sum()),'reference_restore_max_error':0.}
show_source('core/algo/visual_conditioning.py','append_visual_references')


## 3. Semantic time и storage index различаются

Пусть generated RoPE содержит temporal positions 0,…,T−1. Первый reference физически стоит после T generated slots, но получает RoPE t=0. Второй reference — position T−1. Иначе модель сочла бы reference будущим кадром за пределами нужного video.

$$
\mathrm{RoPE}_{extended}=[R_0,\ldots,R_{T-1},R_0,R_{T-1}].
$$

Spatial coordinate rotations refs остаются теми же. `compute_reference_rope` сначала строит generated grid shape(T_total−R,H_p,W_p), затем append; count=0 делегирует обычный compute_visual_rope. Это reference position encoding, не доказательство perfect interpolation между двумя картинками.


In [ ]:
model=tiny_model();shape=(3,2,2);scale=(1.,1.,1.)
base=native('core.algo.prepare_ropes').compute_visual_rope(model.visual_rope,shape,scale)
ext=vc.append_reference_rope(base,2)
computed=vc.compute_reference_rope(model.visual_rope,(5,2,2),scale,2)
plain=vc.compute_reference_rope(model.visual_rope,shape,scale,0)
assert torch.equal(ext,computed) and torch.equal(plain,base)
assert torch.equal(ext[-2],base[0]) and torch.equal(ext[-1],base[-1])
metrics['reference_rope_exact']=True
fig,ax=plt.subplots(figsize=(7,2.8));ax.scatter(range(5),[0,1,2,0,2],s=120,c=['#2078b4']*3+['#e87924']*2)
for i,label in enumerate(['gen0','gen1','gen2','ref first','ref last']):ax.annotate(label,(i,[0,1,2,0,2][i]),xytext=(0,10),textcoords='offset points',ha='center')
ax.set(xlabel='Storage position',ylabel='Semantic temporal position',xticks=range(5),yticks=range(3),ylim=(-.5,3))
fig.tight_layout();fig.savefig(figure_path(LAB,'reference_positions'));plt.show()


## 4. Constraint projection и solver

Reference restoration задаёт projector P на множество states с фиксированными clean refs. При tail scheme после Euler step выполняется:

$$
X_{k+1}=P\left(X_k+\Delta\sigma\,u(X_k,\sigma_k,c)\right).
$$

Это отличается от свободной ODE на всех coordinates. Generated mask нужен при decode/export: reference tail не должен автоматически увеличивать длительность сгенерированного видео. Low-level denoise loop может вернуть extended tensor; filtering принадлежит pipeline/postprocess, который надо проверить отдельно.

I2V first-frame scheme перезаписывает перед network forward и восстанавливает в конце; tail scheme дополнительно restore после каждого update. Это разные алгоритмические места применения ограничения.


In [ ]:
# Projection idempotence is an exact mathematical/control test.
x=batched+torch.randn_like(batched);once=x.clone();vc.restore_tail_references(once,refs,cu_ext)
twice=once.clone();vc.restore_tail_references(twice,refs,cu_ext)
assert torch.equal(once,twice)
metrics['reference_projector_idempotence_error']=float((once-twice).abs().max())
try:vc.append_reference_rope(base,3)
except ValueError:metrics['invalid_reference_count_rejected']=True
else:raise AssertionError('Only one/two references supported by this source API')


## Измерение → вывод

Все пять visual_conditioning methods исполнены. Batched/packed transforms, projector idempotence и semantic RoPE совпали с ожидаемыми indices. Это preprocessing/indexing evidence, не image-to-video generation quality.


## Самостоятельные опыты

1. Выпишите индексы двух refs для каждого B в packed layout.
2. Повторите masks/roles для R=1 и сравните с R=2.
3. Что изменится, если refs получат storage-time RoPE?
4. Нарисуйте места constraint projection в Euler и π sampler.

Перед изменением запишите прогноз. Сохраняйте baseline, меняйте один фактор, отмечайте seed и tensor layout. Нулевая ошибка на oracle не равна pretrained quality.


In [ ]:
result=save_metrics(LAB,metrics)
print(json.dumps(result,ensure_ascii=False,indent=2))


## Первичные источники

[MIT course / notes](https://diffusion.csail.mit.edu/2026/index.html) · [DiT](https://arxiv.org/abs/2212.09748) · [Flow Matching](https://arxiv.org/abs/2210.02747) · [K5](https://arxiv.org/abs/2511.14993) · [K6 §3.2, §4.5](https://arxiv.org/html/2610.05608v1) · [π-Flow §3.1, §4 / Alg.1](https://arxiv.org/html/2510.14974v1) · [Source hashes / provenance](../sources.json).

Формулы ниже/выше выводятся для явно заданных соглашений. Тела source methods сохраняются; adapters и doubles подписаны. Малые experiments не воспроизводят промышленное обучение.
